# Entrenamiento: pronóstico semanal regional + reparto municipal
EDA, features, modelo, comparación contra persistencia/SARIMA, backtest y SHAP.
El pipeline reproducible vive en `model/train.py` (este notebook reutiliza sus funciones).

In [ ]:
import os, sys, json
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
sys.path.insert(0, os.path.abspath('.'))
import train as T
raw = pd.read_parquet(T.DATA)
raw.head()

## 1. EDA: la serie municipal es casi todo ceros; la regional sí tiene señal

In [ ]:
print('% semanas en 0 por municipio-enfermedad (desde 2023-06):')
r = raw[raw.fecha_inicio_semana >= '2023-06-01']
print(r.groupby('enfermedad').casos.apply(lambda s: round((s == 0).mean(), 2)))
region = T.build_region(raw)
fig, axes = plt.subplots(3, 1, figsize=(10, 8), sharex=True)
for ax, (e, g) in zip(axes, region.groupby('enfermedad')):
    ax.plot(g.fecha_inicio_semana, g.casos); ax.set_title(f'Casos semanales región Centro - {e}')
plt.tight_layout(); plt.show()

## 2. Features y partición temporal (sin fuga: se excluyen filas cuyo objetivo cae en test)

In [ ]:
muni = T.build_muni(raw, region)
rall = region.dropna(subset=T.REG_FEATURES + ['rlog_next']).reset_index(drop=True)
mall = muni.dropna(subset=['share', 'casos_next']).reset_index(drop=True)
weeks = np.sort(rall.fecha_inicio_semana.unique())
cut = pd.Timestamp(weeks[int(len(weeks) * (1 - T.TEST_FRAC))])
rtr, rte = rall[rall.fecha_inicio_semana < cut - T.WEEK], rall[rall.fecha_inicio_semana >= cut]
mte = mall[mall.fecha_inicio_semana >= cut]
print('corte test:', cut.date(), '| train', len(rtr), '| test', len(rte))
T.REG_FEATURES

## 3. Hiperparámetros (TimeSeriesSplit solo sobre train) y evaluación en test

In [ ]:
pr, grid = T.tune_region(rtr)
print('mejores hiperparámetros:', pr)
model = T.fit_model(rtr, pr)
th_r = T.thresholds(region[region.fecha_inicio_semana < cut], ['enfermedad'])
th_m = T.thresholds(raw[raw.fecha_inicio_semana < cut], ['municipio_id', 'enfermedad'])
rp, mp = T.predict(model, rte, mte)
ev = T.evaluar(rp, mp, rte, th_r, th_m, sarima=T.sarima_region(region, cut))
pd.DataFrame({k: {'MAE RF': d['rf']['mae'], 'MAE persistencia': d['persistencia']['mae'], 'MAE SARIMA': d['sarima']['mae'],
                  'F1 RF': d['rf']['f1_macro'], 'F1 persistencia': d['persistencia']['f1_macro']}
              for k, d in ev['regional'].items()}).T.round(2)

In [ ]:
pd.DataFrame({k: {'MAE reparto': d['reparto']['mae'], 'MAE persistencia': d['persistencia']['mae'],
                  'F1 reparto': d['reparto']['f1_macro'], 'F1 persistencia': d['persistencia']['f1_macro']}
              for k, d in ev['municipal'].items()}).T.round(3)

In [ ]:
# Matriz de confusión del riesgo regional (RF)
pd.DataFrame(ev['regional']['global']['rf']['matriz_confusion'], index=T.LABELS, columns=T.LABELS)

## 4. Backtest de origen móvil (incluye el brote de dengue de 2024)
Ejecuta `python model/backtest.py`; el resultado queda en `artifacts/backtest.json`.

In [ ]:
bt = json.load(open(os.path.join(T.ART, 'backtest.json'), encoding='utf-8'))
pd.DataFrame({k: {m: v['mae'] for m, v in d.items()} for k, d in bt['agregado'].items()}).T.round(3)

## 5. Explicabilidad (SHAP)
Gráfica generada por `train.py`: `model/docs/shap_summary.png`.

In [ ]:
from IPython.display import Image
Image(os.path.join(T.DOCS, 'shap_summary.png'))

## 6. Conclusiones
- El pronóstico regional supera a la persistencia en dengue (brotes) e influenza; en COVID no.
- El riesgo municipal se deriva de los casos esperados (reparto por participación reciente).
- Limitaciones: conteos pequeños por municipio y sin datos externos (clima/población).